# Python Interview Questions & Answers for Data Scientists

Companion to the [complete article](https://todatabeyond.com/blog/python-interview-questions-and-answers-for-data-scientists). Run all cells in order on Python 3.11+ in Jupyter/Colab. Setup installs pinned NumPy 2.3.3, pandas 2.3.3 and Matplotlib 3.10.9 into the current kernel environment. Only package installation needs a network connection; examples use local, synthetic data, CPU and no API keys.

All nine embedded author gists were recovered completely, including revision IDs, before maintenance. Original snapshots are reproduced in the appendix for provenance, not execution. Fix .intersect to .intersection and preserve the source-required [9, 0] order; avoid shadowing list and reporting duplicates more than once; validate the three-product input; return zero for empty/all-negative subarrays per the original question. Supply missing examples for lists/tuples, Euclidean distance and pandas mapping. Distance compares Series positionally, not by index. The source's first-recurring output n means first repeat encountered scanning left to right, not the first character that eventually repeats.

NumPy normal sampling supplies N and uses a seeded Generator instead of invalid randn((N,)). Transcribe the original grid-density screenshot: correct the density prefactor and make weight normalization explicit, preserve weighted discrete sampling over a finite grid, seed it, and actually honor histogram bins. This is a truncated/discrete approximation, not an exact continuous normal or a claim to implement a PRNG from scratch. Matplotlib supplies both real histograms; no seaborn dependency or KDE is needed.

DataFrame.map replaces deprecated applymap, while apply still demonstrates row/column aggregation. Original public prose remains; apply accepts additional noncallable forms too, and tuple immutability does not freeze mutable contents. Memory and speed comparisons are not universal benchmarks.

Q12 source starts at the top-left and moves only down/right; keep a repaired version with defined recursion, a dead-end base of one and dimension-sized caching instead of MAX=10. Add a separately named general solution for any start/four orthogonal directions, using iterative value-ordered DP to avoid recursion depth. Both return the original example's five. A snake matrix and decreasing matrix distinguish their contracts. The recursive factorial and source-style path remain educational bounded-input functions, not arbitrarily deep recursion guarantees. Hash-based set examples expect hashable values. Git notebook outputs remain empty and execution counts null.


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--disable-pip-version-check', 'numpy==2.3.3',
                'pandas==2.3.3', 'matplotlib==3.10.9'], check=True)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__, '| pandas:', pd.__version__, '| Matplotlib:', matplotlib.__version__)
print('CPU only; no API keys, external data or GPU.')


## 1. Array Intersection


In [ ]:
X = [1, 5, 9, 0]
Y = [3, 0, 2, 9]

def intersection(left, right):
    """Unique shared values, in the order encountered in left."""
    shared = set(left).intersection(right)
    return [value for value in dict.fromkeys(left) if value in shared]

print(intersection(X, Y))


## 2. Find Duplicate Values


In [ ]:
def duplicates(values):
    seen, reported = set(), set()
    result = []
    for value in values:
        if value in seen and value not in reported:
            result.append(value)
            reported.add(value)
        seen.add(value)
    return result

print(duplicates([1, 2, 3, 1, 3, 6, 5]))


## 3. Maximum Product of Three Numbers


In [ ]:
import heapq

def max_three(arr):
    if len(arr) < 3:
        raise ValueError('At least three numbers are required.')
    largest = heapq.nlargest(3, arr)
    smallest = heapq.nsmallest(2, arr)
    return max(largest[0] * largest[1] * largest[2],
               largest[0] * smallest[0] * smallest[1])

print(max_three([1, 5, 3, 4]))
print(max_three([-2, -4, 5, 3]))


## 4. Maximum Contiguous Subarray Sum


In [ ]:
def max_subarray(arr):
    max_sum = curr_sum = 0
    for value in arr:
        curr_sum = max(0, curr_sum + value)
        max_sum = max(max_sum, curr_sum)
    return max_sum

print(max_subarray([0, -1, -5, -2, 3, 14]))
print(max_subarray([-8, -3, -5]))


## 5. Lists and Tuples


In [ ]:
values = [1, 2, 3]
values.append(4)
values[0] = 10
print('Mutable list:', values)

fixed = (1, 2, 3)
try:
    fixed[0] = 10
except TypeError:
    print('Tuple item assignment raises TypeError.')

nested = ([1, 2], 3)
nested[0].append(4)
print('Mutable object inside a tuple:', nested)


## 6. Euclidean Distance Between Series


In [ ]:
from math import dist

def euclidean_distance(left, right):
    """Compare equally sized Series positionally, not by index labels."""
    if len(left) != len(right):
        raise ValueError('Series must have equal lengths.')
    return dist(left.to_numpy(), right.to_numpy())

first = pd.Series([1, 2, 3])
second = pd.Series([4, 6, 3])
print(euclidean_distance(first, second))


## 7. Generate Combinations


In [ ]:
from itertools import combinations

def find_combination(k, n):
    if k < 0 or n < 0:
        raise ValueError('k and n must be nonnegative integers.')
    return [list(values) for values in combinations(range(1, n + 1), k)]

print(find_combination(k=2, n=3))


## 8. Sample a Normal Distribution


In [ ]:
N = 10_000
normal_rng = np.random.default_rng(42)
normal_samples = normal_rng.standard_normal(N)
fig, ax = plt.subplots(figsize=(7, 4))
counts, edges, _ = ax.hist(normal_samples, bins=30, color='#3568b0', edgecolor='white')
ax.set(title='Normal samples: NumPy', xlabel='Value', ylabel='Count')
fig.tight_layout()
print('Samples:', normal_samples.size)
print(f'Mean: {normal_samples.mean():.4f}; std: {normal_samples.std():.4f}')
print('Histogram count:', int(counts.sum()))
plt.show()


### Weighted finite-grid sampling


In [ ]:
def samples_from_normal(n_samples, mean, std, std_range=15,
                        num_of_steps=100_000, seed=42):
    """Sample a finite normal-density grid, not an exact continuous normal."""
    if not isinstance(n_samples, int) or n_samples < 0:
        raise ValueError('n_samples must be a nonnegative integer.')
    if not isinstance(num_of_steps, int) or num_of_steps <= 0:
        raise ValueError('num_of_steps must be a positive integer.')
    if not all(np.isfinite(value) for value in (mean, std, std_range)):
        raise ValueError('Distribution parameters must be finite.')
    if std <= 0 or std_range <= 0:
        raise ValueError('std and std_range must be positive.')
    min_x = mean - std_range * std
    max_x = mean + std_range * std
    x_values = np.linspace(min_x, max_x, num_of_steps, endpoint=False)
    max_value = 1 / (std * np.sqrt(2 * np.pi))
    weights = max_value * np.exp(-0.5 * ((x_values - mean) / std) ** 2)
    weights = weights / weights.sum()
    rng = np.random.default_rng(seed)
    return rng.choice(x_values, n_samples, p=weights)

def plot_normal_sample_hist(n_samples, mean, std, bins=30):
    samples = samples_from_normal(n_samples, mean, std)
    fig, ax = plt.subplots(figsize=(7, 4))
    counts, edges, _ = ax.hist(samples, bins=bins, color='#3568b0', edgecolor='white')
    ax.set(title='Normal samples: weighted finite grid', xlabel='Value', ylabel='Count')
    fig.tight_layout()
    print('Samples:', samples.size)
    print(f'Mean: {samples.mean():.4f}; std: {samples.std():.4f}')
    print('Histogram count:', int(counts.sum()))
    plt.show()
    return samples, counts

grid_samples, grid_counts = plot_normal_sample_hist(10_000, 0, 1)


## 9. Apply and Element-Wise Mapping


In [ ]:
frame = pd.DataFrame({'a': [1, 2], 'b': [3, 4]})
print('Column sums (apply):')
print(frame.apply(sum, axis=0).to_string())
print('Row sums (apply):')
print(frame.apply(sum, axis=1).to_string())
print('Element-wise doubling (map):')
mapped = frame.map(lambda value: value * 2)
print(mapped.to_string())


## 10. First Recurring Character


In [ ]:
def first_recurring(input_str):
    seen = set()
    for letter in input_str:
        if letter in seen:
            return letter
        seen.add(letter)
    return None

print(first_recurring('pythoninterviewquestion'))
print(first_recurring('abc'))


## 11. Recursive Factorial


In [ ]:
def factorial(x):
    if not isinstance(x, int) or isinstance(x, bool):
        raise TypeError('x must be an integer.')
    if x < 0:
        return -1
    if x <= 1:
        return 1
    return x * factorial(x - 1)

print(factorial(5))
print(factorial(0))
print(factorial(-1))


## 12. Longest Increasing Matrix Paths


In [ ]:
from functools import cache

def longest_increasing_path_from_top_left(matrix):
    """Start at (0, 0); move only down/right to a strictly larger value."""
    if not matrix:
        return 0
    rows, cols = len(matrix), len(matrix[0])
    if any(len(row) != cols for row in matrix):
        raise ValueError('Matrix must be rectangular.')
    if not cols:
        return 0

    @cache
    def visit(row, col):
        result = 1
        for next_row, next_col in ((row + 1, col), (row, col + 1)):
            if (next_row < rows and next_col < cols
                    and matrix[next_row][next_col] > matrix[row][col]):
                result = max(result, 1 + visit(next_row, next_col))
        return result

    return visit(0, 0)

matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print('Down/right from top-left:', longest_increasing_path_from_top_left(matrix))


### Any start, four orthogonal directions


In [ ]:
def longest_increasing_path(matrix):
    """Start anywhere; use strictly increasing orthogonal moves, no diagonals."""
    if not matrix:
        return 0
    rows, cols = len(matrix), len(matrix[0])
    if any(len(row) != cols for row in matrix):
        raise ValueError('Matrix must be rectangular.')
    if not cols:
        return 0
    lengths = [[1] * cols for _ in range(rows)]
    cells = sorted((matrix[row][col], row, col)
                   for row in range(rows) for col in range(cols))
    for value, row, col in cells:
        for next_row, next_col in ((row - 1, col), (row + 1, col),
                                   (row, col - 1), (row, col + 1)):
            if (0 <= next_row < rows and 0 <= next_col < cols
                    and matrix[next_row][next_col] < value):
                lengths[row][col] = max(lengths[row][col],
                                        1 + lengths[next_row][next_col])
    return max(map(max, lengths))

print('Any start, four directions:', longest_increasing_path(matrix))
print('Snake matrix:', longest_increasing_path([[1, 2, 3], [6, 5, 4], [7, 8, 9]]))


## Regression checks


In [ ]:
import itertools
import math
import random
from collections import Counter

def raises(error, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except error:
        return
    raise AssertionError('Expected ' + error.__name__)

assert intersection(X, Y) == [9, 0]
assert intersection([3, 3, 1, 2, 1], [1, 3]) == [3, 1]
assert intersection([], [1]) == []
assert duplicates([1, 1, 1, 2, 2, 2]) == [1, 2]
assert duplicates([]) == []
array_cases = 0
for size in range(7):
    for values in itertools.product((-2, 0, 3), repeat=size):
        expected_duplicates = [v for v, count in Counter(values).items() if count > 1]
        assert set(duplicates(values)) == set(expected_duplicates)
        assert len(duplicates(values)) == len(expected_duplicates)
        expected_sum = max([0] + [sum(values[start:end])
                           for start in range(size) for end in range(start + 1, size + 1)])
        assert max_subarray(values) == expected_sum
        if size >= 3:
            assert max_three(values) == max(math.prod(c) for c in itertools.combinations(values, 3))
        else:
            raises(ValueError, max_three, values)
        array_cases += 1
assert max_three([-10, -9, -8, -7]) == -504
assert max_subarray([-1, -2]) == 0
assert euclidean_distance(first, second) == 5.0
assert euclidean_distance(pd.Series([], dtype=float), pd.Series([], dtype=float)) == 0.0
assert euclidean_distance(pd.Series([1, 2], index=['a', 'b']),
                          pd.Series([4, 6], index=['b', 'a'])) == 5.0
raises(ValueError, euclidean_distance, pd.Series([1]), pd.Series([1, 2]))
for n in range(9):
    for k in range(n + 3):
        result = find_combination(k, n)
        assert len(result) == math.comb(n, k)
        assert result == [list(c) for c in itertools.combinations(range(1, n + 1), k)]
raises(ValueError, find_combination, -1, 3)
raises(ValueError, find_combination, 1, -1)
assert normal_samples.shape == grid_samples.shape == (10_000,)
assert int(counts.sum()) == int(grid_counts.sum()) == 10_000
assert np.array_equal(normal_samples, np.random.default_rng(42).standard_normal(10_000))
assert np.array_equal(grid_samples, samples_from_normal(10_000, 0, 1))
assert abs(normal_samples.mean()) < 0.05 and abs(normal_samples.std() - 1) < 0.05
assert abs(grid_samples.mean()) < 0.05 and abs(grid_samples.std() - 1) < 0.05
shifted = samples_from_normal(20_000, 5, 2)
assert abs(shifted.mean() - 5) < 0.1 and abs(shifted.std() - 2) < 0.1
assert samples_from_normal(0, 0, 1).size == 0
assert np.all((-15 <= grid_samples) & (grid_samples < 15))
for kwargs in ({'std': 0}, {'std': -1}, {'std_range': 0}, {'std_range': -1},
               {'std': float('nan')}, {'mean': float('inf')}, {'num_of_steps': 0}):
    arguments = dict(n_samples=1, mean=0, std=1); arguments.update(kwargs)
    raises(ValueError, samples_from_normal, **arguments)
raises(ValueError, samples_from_normal, -1, 0, 1)
pd.testing.assert_frame_equal(mapped, frame * 2)
assert frame.apply(sum).tolist() == [3, 7]
assert frame.apply(sum, axis=1).tolist() == [4, 6]
for value in ('', 'a', 'abc', 'abca', 'abba', 'éaé', 'pythoninterviewquestion'):
    expected = next((c for i, c in enumerate(value) if c in value[:i]), None)
    assert first_recurring(value) == expected
for value in range(101):
    assert factorial(value) == math.factorial(value)
assert factorial(-2) == -1
raises(TypeError, factorial, 1.5)
raises(TypeError, factorial, True)

def brute_path(mat, directions, all_starts):
    if not mat or not mat[0]:
        return 0
    rows, cols = len(mat), len(mat[0])
    def explore(row, col):
        following = [(r, c) for dr, dc in directions
                     for r, c in [(row + dr, col + dc)]
                     if 0 <= r < rows and 0 <= c < cols and mat[r][c] > mat[row][col]]
        return 1 + max([explore(r, c) for r, c in following], default=0)
    starts = itertools.product(range(rows), range(cols)) if all_starts else [(0, 0)]
    return max(explore(r, c) for r, c in starts)

rng = random.Random(42)
matrices = [[], [[]], [[7]], [[2, 2], [2, 2]], [[9, 8], [7, 6]], matrix,
            [[1, 2, 3], [6, 5, 4], [7, 8, 9]], [[9, 9, 4], [6, 6, 8], [2, 1, 1]]]
matrices += [[[rng.randrange(1, 10) for _ in range(rng.randrange(1, 5))]] for _ in range(5)]
for _ in range(40):
    rows, cols = rng.randrange(1, 5), rng.randrange(1, 5)
    matrices.append([[rng.randrange(1, 10) for _ in range(cols)] for _ in range(rows)])
for mat in matrices:
    assert longest_increasing_path_from_top_left(mat) == brute_path(mat, [(1, 0), (0, 1)], False)
    assert longest_increasing_path(mat) == brute_path(mat, [(-1, 0), (1, 0), (0, -1), (0, 1)], True)
assert longest_increasing_path_from_top_left([[9, 8], [7, 6]]) == 1
assert longest_increasing_path([[9, 8], [7, 6]]) == 3
assert longest_increasing_path([list(range(1500))]) == 1500
assert longest_increasing_path_from_top_left([[r * 11 + c for c in range(11)] for r in range(11)]) == 21
raises(ValueError, longest_increasing_path, [[1], [2, 3]])
raises(ValueError, longest_increasing_path_from_top_left, [[], [2]])
print(f'All 12 questions and 14 example cells passed; {array_cases} exhaustive array fixtures checked.')
print('Combination counts, factorials 0–100, positional distances, duplicates and Unicode recurrence passed.')
print('Both seeded 10,000-sample histograms rendered; counts, moments, support and validation passed.')
print(f'Both matrix algorithms matched independent exhaustive searches on {len(matrices)} fixtures.')
print('General path handles 1,500 cells without recursion; source-style path supports matrices larger than 10×10.')


## Original recovered gist snapshots (not executed)

### Q1

[0293c9197d2ac14658716be0afb87414](https://gist.github.com/youssefHosni/0293c9197d2ac14658716be0afb87414) · revision `cba89e92ab0f8f7b88b52be143f1cb9df78e5d83`

```python
X = [1,5,9,0] 
Y = [3,0,2,9]
asnwer = set(X).intersect (set(Y))
```

### Q2

[f276ad81ae44ef3919722c1e2a94810d](https://gist.github.com/youssefHosni/f276ad81ae44ef3919722c1e2a94810d) · revision `b05f00d930d43055c0265e38cbaa74d952142baa`

```python
list = [1,2,3,1,3,6,5]
set1=set()
res=[]
for i in list:
  if i in set1:
    res.append(i)
  else:
    set1.add(i)
print(res)
```

### Q3

[dd40b6771cecca7287b0a21dac615bc3](https://gist.github.com/youssefHosni/dd40b6771cecca7287b0a21dac615bc3) · revision `cb7d009b7355fed9f6013df0b907a73b98831664`

```python
import heapq
def max_three(arr):
    a = heapq.nlargest(3, arr) # largerst 3 numbers for postive case
    b = heapq.nsmallest(2, arr) # for negative case
    return max(a[2]*a[1]*a[0], b[1]*b[0]*a[0])
```

### Q4

[a1d4d4a01c0dc0e75cf0012c90dfcec8](https://gist.github.com/youssefHosni/a1d4d4a01c0dc0e75cf0012c90dfcec8) · revision `53cbc72c4e9b7b8e38ed438ed3251e6109e9994c`

```python
def max_subarray(arr):
  n = len(arr)
  max_sum = arr[0] #max
  curr_sum = 0 
  for i in range(n):
    curr_sum += arr[i]
    max_sum = max(max_sum, curr_sum)
    if curr_sum <0:
      curr_sum  = 0
  return max_sum    
      
```

### Q7

[4fc3da7e345071253a44c357332a1a07](https://gist.github.com/youssefHosni/4fc3da7e345071253a44c357332a1a07) · revision `a084b07aa98c0f1dedba06a9f95fe2aed3b512f0`

```python
from itertools import combinations
def find_combintaion(k,n):
    list_num = []
    comb = combinations([x for x in range(1, n+1)],k)
    for i in comb:
        list_num.append(i)
    print("(K:{},n:{}):".format(k,n))
    print(list_num,"\n")
```

### Q8

[8d85a7004983e80e1399738a3ce0042b](https://gist.github.com/youssefHosni/8d85a7004983e80e1399738a3ce0042b) · revision `c0d04ef1d6306e168787c8bc342545c3423bc644`

```python
import numpy as np
import matplotlib.pyplot as plt

x = np.random.randn((N,))
plt.hist(x)
```

### Q10

[4b826ad78be3f3d12bc740c561b8dcc1](https://gist.github.com/youssefHosni/4b826ad78be3f3d12bc740c561b8dcc1) · revision `459b13ee4a40276bc9e8c5d67ce550dee6350d87`

```python
input_string = "pythoninterviewquestion"

def first_recurring(input_str):
  
  a_str = ""
  for letter in input_str:
    a_str = a_str + letter
    if a_str.count(letter) > 1:
      return letter
  return None

first_recurring(input_string)

```

### Q11

[d98c2f42a8197dd62cb9090bd54bd069](https://gist.github.com/youssefHosni/d98c2f42a8197dd62cb9090bd54bd069) · revision `980bdd628a095d6024f95b47dd8838cffcc616f0`

```python
def factorial(x):
    # Edge cases
    if x < 0: return -1
    if x == 0: return 1
    
    # Exit condition - x = 1
    if x == 1:
        return x
    else:
        # Recursive part
        return x * factorial(x - 1)
    

```

### Q12

[42f5f0b08a91eb7cb1a48799fd9fe82d](https://gist.github.com/youssefHosni/42f5f0b08a91eb7cb1a48799fd9fe82d) · revision `4fb0566d23de5b90d8c6c6285abe1991ffb22249`

```python
MAX = 10

def Longest_Increasing_Path(dp, mat, n, m, x, y):
     
    # If value not calculated yet.
    if (dp[x][y] < 0):
        result = 0
         
        #  // If reach bottom right cell, return 1
        if (x == n - 1 and y == m - 1):
            dp[x][y] = 1
            return dp[x][y]
 
        # If reach the corner
        # of the matrix.
        if (x == n - 1 or y == m - 1):
            result = 1
 
        # If value greater than below cell.
        if (x + 1 < n and mat[x][y] < mat[x + 1][y]):
            result = 1 + LIP(dp, mat, n,
                            m, x + 1, y)
 
        # If value greater than left cell.
        if (y + 1 < m and mat[x][y] < mat[x][y + 1]):
            result = max(result, 1 + LIP(dp, mat, n,
                                        m, x, y + 1))
        dp[x][y] = result
    return dp[x][y]
 
# Wrapper function
def wrapper(mat, n, m):
    dp = [[-1 for i in range(MAX)]
            for i in range(MAX)]
    return Longest_Increasing_Path(dp, mat, n, m, 0, 0)

```

### Original code-image transcription (Q8)

```python
def samples_from_normal(n_samples, mean, std, std_range=15, num_of_steps=100000):
    max_value = 1/std*np.sqrt(2*np.pi)
    min_x = mean - std_range*std
    max_x = mean + std_range*std
    step = (max_x-min_x)/num_of_steps
    x_values = []
    weights = []
    for value in np.arange(min_x,max_x,step):
        exp_f = np.exp(-0.5*((value-mean)/std)**2)
        x_values.append(value)
        weights.append(max_value*exp_f)
    weights = weights/sum(weights)
    return np.random.choice(x_values, n_samples, p=weights)

def plot_normal_sample_hist(n_samples, mean, std, bins=30):
    sns.histplot(data=pd.DataFrame(samples_from_normal(n_samples, mean, std)), kde=True)
```
